# Analyse Exploratoire des Données (EDA) - Archelec

Ce notebook répond aux exigences d'analyse de données (EDA) du projet. Il est divisé en deux sections :
1. **Analyse générale du corpus** : volumétrie, nettoyage, longueur des textes et intégrité des métadonnées.
2. **Analyse ciblée pour la génération** : validation de la variable conditionnelle `famille_politique` par l'analyse du style (verbosité) et des spécificités lexicales (TF-IDF).

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from pathlib import Path
import re
from sklearn.feature_extraction.text import TfidfVectorizer

# Configuration visuelle des graphiques
sns.set_theme(style="whitegrid")

ModuleNotFoundError: No module named 'matplotlib'

## 1. Chargement et Nettoyage des Données

In [ ]:
# --- Fichiers texte à charger ---
PATTERNS = [
    "data/1981/legislatives/*PF*.txt",
    "data/1988/legislatives/*PF*.txt",
    "data/1993/legislatives/*PF*.txt",
]

files = []
for p in PATTERNS:
    files.extend(sorted(Path().glob(p)))

dataset = load_dataset("text", data_files=PATTERNS, split="train", sample_by="document")
print(f"Dataset brut chargé : {len(dataset)} documents")

# --- Métadonnées ---
metadata = pd.read_csv("data/archelect_search.csv")
COLS_META = ["id", "titulaire-prenom", "titulaire-nom", "titulaire-profession",
             "titulaire-soutien", "contexte-tour", "date", "departement-nom"]
metadata_dict = metadata[COLS_META].set_index("id").to_dict("index")

def add_metadata(example, idx):
    path = files[idx]
    doc_id = path.stem
    example["id"] = doc_id
    example["annee"] = path.parts[-3]

    meta = metadata_dict.get(doc_id, {})
    example["prenom"]     = meta.get("titulaire-prenom", "non mentionné")
    example["nom"]        = meta.get("titulaire-nom", "non mentionné")
    example["profession"] = meta.get("titulaire-profession", "non mentionné")
    example["soutien"]    = meta.get("titulaire-soutien", "non mentionné")
    example["tour"]       = meta.get("contexte-tour", "non mentionné")
    example["date"]       = meta.get("date", "non mentionné")
    example["departement"]= meta.get("departement-nom", "non mentionné")
    return example

dataset = dataset.map(add_metadata, with_indices=True)
df = dataset.to_pandas()

In [ ]:
# Nettoyage OCR (Reprise des Regex du notebook principal)
_RE_HYPHENATION = re.compile(r'([A-Za-zÀ-ÿ]+)-\s+([A-Za-zÀ-ÿ]+)')
_RE_ISOLATED_CAP = re.compile(r'\b([A-Z])\s+([A-Z]+\b)')
_RE_WATERMARK  = re.compile(r'Sciences Po / fonds CEVIPOF|[☐☒@¥]')
_RE_VU_CAND    = re.compile(r'vu\s*[,:\-]?\s*(le|la|les)\s+candidat[e]?[s]?\s*[:.]?', re.IGNORECASE)
_RE_DROP_LINE  = re.compile(r'^.*(imp\.?\s|imprimerie|imprimeurs|r\.?c\.?\s|\b\d{5}\b|\b\d{1,2}([\s.\-]?\d{2}){3}\b).*$', re.IGNORECASE | re.MULTILINE)
_RE_MULTILINE  = re.compile(r'\n{3,}')
_RE_MULTSPACE  = re.compile(r' {2,}')
_RE_ENUM       = re.compile(r'[:\n]\s*[•*>.o·]\s*', re.MULTILINE)

def clean_text(text):
    text = str(text)
    text = _RE_HYPHENATION.sub(r'\1\2', text)
    text = _RE_ISOLATED_CAP.sub(r'\1\2', text)
    text = _RE_WATERMARK.sub("", text)          
    text = _RE_VU_CAND.sub("", text)            
    text = _RE_DROP_LINE.sub("", text)          
    text = _RE_MULTILINE.sub('\n\n', text)      
    text = _RE_MULTSPACE.sub(' ', text)         
    text = _RE_ENUM.sub("- ", text)             
    return text.strip()

df['text_clean'] = df['text'].apply(clean_text)
df['word_count'] = df['text_clean'].apply(lambda x: len(x.split()))
print("Nettoyage OCR terminé.")

## 2. Analyse Générale du Corpus

On s'intéresse ici à la qualité des données qui seront injectées dans notre LLM : y a-t-il beaucoup de métadonnées manquantes ? Quelle est la longueur moyenne d'un discours politique ?

In [ ]:
# Analyse des données manquantes (conditionnement)
missing_data = df[['prenom', 'nom', 'profession', 'soutien', 'departement']].apply(lambda x: (x == 'non mentionné').sum() / len(df) * 100)

plt.figure(figsize=(8, 4))
sns.barplot(x=missing_data.values, y=missing_data.index, palette='Reds_r')
plt.title('Pourcentage de valeurs manquantes par métadonnée')
plt.xlabel('% Manquant')
plt.show()

print("Ce graphique est crucial : on remarque que les métadonnées (qui vont servir de prompt) sont très bien renseignées, justifiant leur utilisation comme input pour la génération.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Répartition par année
sns.countplot(data=df, x='annee', ax=axes[0], palette='Blues_d')
axes[0].set_title('Nombre de documents par année électorale')
axes[0].set_ylabel('Nombre de documents')

# Distribution de la verbosité (Nombre de mots)
sns.histplot(data=df, x='word_count', bins=50, ax=axes[1], color='purple', kde=True)
axes[1].set_title('Distribution du nombre de mots par profession de foi')
axes[1].set_xlabel('Nombre de mots')
axes[1].axvline(df['word_count'].median(), color='red', linestyle='--', label=f"Médiane : {df['word_count'].median():.0f}")
axes[1].legend()

plt.tight_layout()
plt.show()

## 3. Analyse Ciblée : L'impact de la Famille Politique

Puisque la tâche consiste à générer un texte conditionné sur l'étiquette politique, nous devons valider linguistiquement que l'appartenance à un parti modifie réellement la rhétorique.

In [ ]:
# On utilise ta fonction de normalisation pour agréger les dizaines de petits partis
def normalize_party(soutien):
    if pd.isna(soutien) or soutien == "non mentionné":
        return "A_EXCLURE"
    s = str(soutien).lower().split(';')[0].strip()
    if any(x in s for x in ["front national", "fn", "extrême droite", "trop d'immigrés", "nationaliste", "royaliste", "action française", "forces nouvelles"]):
        return "Extreme_Droite"
    if any(x in s for x in ["communiste", "pcf", "lutte ouvrière", "lcr", "marxiste", "trotskyste", "parti des travailleurs", "rouge et vert", "psu", "parti socialiste unifié", "combat ouvrier", "alternative démocratie socialisme"]):
        return "Communiste_et_Extreme_Gauche"
    if any(x in s for x in ["écolog", "ecolog", "vert", "environnement", "amis de la terre", "nature et animaux", "biosphère"]) and "chasse" not in s:
        return "Ecologiste"
    if any(x in s for x in ["socialiste", "ps", "mrg", "radicaux de gauche", "majorité présidentielle", "gauche progressiste", "convention des institutions républicaines"]):
        return "Socialiste"
    if any(x in s for x in ["rpr", "rassemblement pour la république", "gaulliste", "cni", "indépendants et paysans", "parti républicain", "droite", "républicain indépendant", "mouvement pour la france"]):
        return "Droite"
    if any(x in s for x in ["udf", "union pour la démocratie française", "centre", "cds", "centriste", "démocratie chrétienne", "parti radical", "radicaux-socialistes", "réformateurs"]):
        return "Centre"
    if any(x in s for x in ["chasse", "cpnt", "rurale"]):
        return "Droite"
    if any(x in s for x in ["corse", "corsica", "kanak", "polynési", "breton", "emgann", "occitan", "catalan", "savoie", "abertzale", "guadeloup", "martiniqu", "indépendantiste", "taatiraa", "tahoeraa"]):
        return "Regionaliste"
    if any(x in s for x in ["sans étiquette", "apolitique", "hors des partis", "indépendant", "libre", "aucun parti", "sans appartenance"]):
        return "Sans_Etiquette"
    return "A_EXCLURE"

df['famille_politique'] = df['soutien'].apply(normalize_party)
df_cibled = df[df['famille_politique'] != "A_EXCLURE"].copy()

plt.figure(figsize=(10, 5))
order = df_cibled['famille_politique'].value_counts().index
sns.countplot(data=df_cibled, y='famille_politique', order=order, palette='Set2')
plt.title('Répartition des documents par famille politique')
plt.xlabel('Nombre de documents')
plt.ylabel('')
plt.show()

### A. Analyse du Style : La verbosité par parti
On cherche à savoir si certaines familles politiques ont tendance à produire des textes structurellement plus longs.

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(data=df_cibled, x='famille_politique', y='word_count', order=order, palette='Set2')
plt.title('Verbosité (Nombre de mots) par famille politique')
plt.xlabel('')
plt.ylabel('Nombre de mots')
plt.xticks(rotation=45, ha='right')
plt.ylim(0, df_cibled['word_count'].quantile(0.95) * 1.2)
plt.show()

print("Analyse : Une différence dans les médianes et l'amplitude des textes justifie l'intérêt d'un LLM capable d'adapter le style et la longueur de la réponse générée.")

### B. Analyse Lexicale : Champs lexicaux spécifiques (TF-IDF)
En calculant le TF-IDF moyen, on peut extraire la fameuse "Langue de Bois" (The Wooden Language) spécifique à chaque parti. C'est l'argument principal pour entraîner un modèle génératif.

In [ ]:
# On se concentre sur 4 grandes familles pour une visualisation claire
familles_cibles = ["Communiste_et_Extreme_Gauche", "Socialiste", "Droite", "Extreme_Droite"]
df_tfidf = df_cibled[df_cibled['famille_politique'].isin(familles_cibles)].copy()

# TF-IDF : max_df=0.8 ignore les "stop words" politiques globaux (ex: France, République, candidat)
# min_df=0.05 assure qu'un mot est quand même assez fréquent pour être un vrai trait de style
vectorizer = TfidfVectorizer(max_df=0.8, min_df=0.05, stop_words='french', max_features=1000)
tfidf_matrix = vectorizer.fit_transform(df_tfidf['text_clean'])
feature_names = vectorizer.get_feature_names_out()

# Calcul du score TF-IDF moyen par parti
df_tfidf_scores = pd.DataFrame(tfidf_matrix.toarray(), columns=feature_names)
df_tfidf_scores['famille_politique'] = df_tfidf['famille_politique'].values
mean_tfidf_by_party = df_tfidf_scores.groupby('famille_politique').mean()

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Mots les plus spécifiques (TF-IDF moyen) par famille politique', fontsize=18)

for ax, famille in zip(axes.flatten(), familles_cibles):
    top_words = mean_tfidf_by_party.loc[famille].sort_values(ascending=False).head(10)
    sns.barplot(x=top_words.values, y=top_words.index, ax=ax, palette='viridis')
    ax.set_title(famille)
    ax.set_xlabel('Score TF-IDF moyen')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

**Conclusion de l'analyse exploratoire :**
1. Les données contextuelles sont très complètes (peu de valeurs manquantes).
2. Il y a une disparité claire dans le vocabulaire utilisé. Générer un texte "neutre" ne reflèterait pas la réalité politique.
3. Ces résultats justifient le fine-tuning d'un LLM sur la variable `famille_politique` pour conditionner la génération sémantique et stylistique du texte.